# Suffix Pruning Visualization (GSM8K)

在 warm-up 阶段裁剪 suffix，对比 3 种模式的 denoising 过程：

| 模式 | warm-up 输入 | 说明 |
|------|-------------|------|
| **full** | 完整序列 | baseline |
| **tail** | prefix + block + 1 tail token | 最小 suffix |
| **window** | prefix + block + 2×block_length + tail | 保留近邻窗口 |

每种模式跑 50 个 GSM8K sample，生成 HTML 可视化文件。

## 1. 环境设置

In [ ]:
import os, sys, gc, random
import torch
import torch.nn.functional as F
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

## 3. 构建 prompt（5-shot，前 50 题）

In [ ]:
import re

FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
questions = [gsm8k[i]['question'] for i in range(LIMIT)]
ref_answers = [gsm8k[i]['answer'] for i in range(LIMIT)]
print(f'Built {len(prompts)} prompts, first length: {prompts[0].shape[1]} tokens')

## 4. 配置

In [ ]:
GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.9
WINDOW_BLOCKS = 2   # window 模式保留的额外 block 数

MODES = ['full', 'tail', 'window']

print(f'Modes: {MODES}')
print(f'gen_length={GEN_LENGTH}, steps={STEPS}, block_length={BLOCK_LENGTH}')
print(f'window 模式: 保留 {WINDOW_BLOCKS}×{BLOCK_LENGTH}={WINDOW_BLOCKS*BLOCK_LENGTH} tokens 后缀 + tail')
print(f'Samples per mode: {LIMIT}')

## 5. 生成函数（1-forward suffix pruning + history 收集）

每个 block **只需 1 次 forward**：

1. **裁剪 forward** — 输入 pruned 序列 + `position_ids` → logits + KV cache
2. **KV trim** — 去掉 block 的 KV 条目，保留 prefix + suffix 作为 context_kv
3. **Refine** — 每步喂 block tokens + context_kv + `position_ids`（不用 replace_position）

KV cache 存的是 **不带 RoPE 的 K/V**，每次 forward 根据 `position_ids` 重新施加 RoPE，
因此即使 context_kv 的位置不连续也能正确工作。

In [ ]:
@torch.no_grad()
def generate_with_collection_prune(
    model, prompt,
    steps=256, gen_length=256, block_length=32,
    temperature=0.0, remasking='low_confidence',
    mask_id=126336, threshold=0.9,
    suffix_mode='full',
    window_blocks=2,
):
    """
    1-forward suffix pruning generation with history collection.

    Warm-up: 1 pruned forward → logits + KV cache
    KV trim: remove block entries, keep prefix + suffix
    Refine:  feed block with context_kv + position_ids (no replace_position)
    """
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    seq_len = Lp + gen_length
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    x = torch.full((B, seq_len), mask_id, dtype=torch.long, device=model.device)
    x[:, :Lp] = prompt
    nfe = 0
    history = []
    pconf = torch.zeros(B, seq_len, dtype=torch.float64, device=model.device)

    def _gumbel(logits):
        if temperature == 0:
            return logits
        lg = logits.to(torch.float64)
        noise = torch.rand_like(lg, dtype=torch.float64)
        return lg.exp() / ((-torch.log(noise)) ** temperature)

    def _compute_x0_conf(logits, mask):
        x0 = torch.argmax(_gumbel(logits), dim=-1)
        if remasking == 'low_confidence':
            p = F.softmax(logits.to(torch.float64), dim=-1)
            conf = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
        else:
            conf = torch.rand(x0.shape, device=x0.device, dtype=torch.float64)
        return x0, conf

    def _transfer(logits, mask, x_cur, quota):
        x0, conf = _compute_x0_conf(logits, mask)
        x0 = torch.where(mask, x0, x_cur)
        neg = torch.tensor(torch.finfo(conf.dtype).min, device=conf.device, dtype=conf.dtype)
        cval = torch.where(mask, conf, neg)
        if threshold is not None:
            tidx = mask & (cval >= threshold)
            mx = torch.argmax(cval, dim=1, keepdim=True)
            force = torch.zeros_like(tidx).scatter_(1, mx, True) & mask
            tidx = tidx | force
        else:
            _, idx = torch.sort(cval, dim=1, descending=True)
            cols = torch.arange(cval.shape[1], device=cval.device).unsqueeze(0)
            tidx = (cols < quota.unsqueeze(1))
            scatter_t = torch.zeros_like(tidx, dtype=torch.int8)
            scatter_t = scatter_t.scatter(1, idx, tidx.to(torch.int8)).bool() & mask
            tidx = scatter_t
        return x0, conf, tidx

    def _snap(step_i, nb, s, e, x_now, x0_full, conf_full):
        mask_now = (x_now == mask_id)
        pconf[:, s:e] = conf_full[:, s:e] if conf_full.shape[1] > (e - s) else conf_full
        return {
            'step': step_i, 'block_id': nb,
            'x': x_now.cpu().numpy().copy(),
            'x0': x0_full.cpu().numpy().copy(),
            'mask': mask_now.cpu().numpy().copy(),
            'confidence': pconf.cpu().numpy().copy(),
            'current_block_range': (s, e),
        }

    def _get_num_transfer_tokens(block_mask, n_steps):
        total = block_mask.sum(dim=1)
        base = torch.div(total, n_steps, rounding_mode='floor')
        rem = total - base * n_steps
        ntt = base.unsqueeze(1).expand(-1, n_steps).clone()
        cols = torch.arange(n_steps, device=block_mask.device).unsqueeze(0)
        ntt = ntt + (cols < rem.unsqueeze(1)).long()
        return ntt

    def _build_keep_indices(e_pos):
        """Build kept position indices based on suffix_mode."""
        if suffix_mode == 'full':
            return list(range(seq_len))
        keep = list(range(e_pos))
        if suffix_mode == 'window':
            window_end = min(e_pos + window_blocks * block_length, seq_len)
            keep.extend(range(e_pos, window_end))
        if keep[-1] != seq_len - 1:
            keep.append(seq_len - 1)
        return keep

    def _trim_kv(past_kv, remove_start, remove_end):
        """Remove KV entries at indices [remove_start, remove_end) from cache."""
        trimmed = []
        for layer_kv in past_kv:
            layer_trimmed = []
            for kv_t in layer_kv:
                prefix_part = kv_t[:, :, :remove_start, :]
                suffix_part = kv_t[:, :, remove_end:, :]
                layer_trimmed.append(torch.cat([prefix_part, suffix_part], dim=2))
            trimmed.append(tuple(layer_trimmed))
        return trimmed

    for nb in range(num_blocks):
        s = Lp + nb * block_length
        e = s + block_length
        block_steps = []

        bmask = (x[:, s:e] == mask_id)
        ntt = _get_num_transfer_tokens(bmask, steps_per_block)

        # ---- Build pruned input ----
        keep_idx = _build_keep_indices(e)
        keep_t = torch.tensor(keep_idx, device=x.device, dtype=torch.long)
        x_prun = x[:, keep_t]
        warmup_pos_ids = keep_t.unsqueeze(0).expand(B, -1)

        # ---- Warm-up: 1 forward on pruned input ----
        out = model(x_prun, use_cache=True, position_ids=warmup_pos_ids)
        nfe += 1

        # Block logits are at indices s:e in x_prun (prefix+block is contiguous)
        block_logits = out.logits[:, s:e, :]
        past_kv = out.past_key_values
        del out

        # ---- Trim KV: remove block entries (indices s..e-1), keep prefix+suffix ----
        # KV layout mirrors keep_idx: [prefix 0..s-1 | block s..e-1 | suffix ...]
        context_kv = _trim_kv(past_kv, s, e)
        del past_kv

        # ---- Build position_ids for refine ----
        # context_kv positions: prefix [0..s-1] + suffix keep_idx[e:]
        # After cat in attention: [context_kv | new_block_kv]
        # position_ids must cover all K entries
        context_positions = keep_idx[:s] + keep_idx[e:]
        block_positions = list(range(s, e))
        refine_pos_ids = torch.tensor(
            context_positions + block_positions,
            device=x.device, dtype=torch.long,
        ).unsqueeze(0).expand(B, -1)

        # ---- Step 0: transfer using warm-up block logits ----
        block_mask_0 = (x[:, s:e] == mask_id)
        quota0 = None if threshold is not None else ntt[:, 0]
        x0_blk, conf_blk, tidx_blk = _transfer(block_logits, block_mask_0, x[:, s:e], quota0)
        del block_logits

        # Build full-sequence snapshot
        x0_full = x.clone()
        x0_full[:, s:e] = x0_blk
        conf_full = pconf.clone()
        conf_full[:, s:e] = conf_blk

        # Apply transfer
        blk_new = torch.where(tidx_blk, x0_blk, x[:, s:e])
        x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)
        block_steps.append(_snap(0, nb, s, e, x, x0_full, conf_full))

        # ---- Refine steps: block input + context_kv + position_ids ----
        for si in range(1, steps_per_block):
            if (x[:, s:e] == mask_id).sum() == 0:
                break

            logits_blk = model(
                x[:, s:e],
                past_key_values=context_kv,
                use_cache=False,
                position_ids=refine_pos_ids,
            ).logits

            mask_blk = (x[:, s:e] == mask_id)
            quota_i = None if threshold is not None else ntt[:, si]
            x0_blk, conf_blk, tidx_blk = _transfer(logits_blk, mask_blk, x[:, s:e], quota_i)

            blk_new = torch.where(tidx_blk, x0_blk, x[:, s:e])
            x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)

            x0_snap = x.clone()
            x0_snap[:, s:e] = torch.where(mask_blk, x0_blk, x[:, s:e])
            pconf[:, s:e] = torch.where(mask_blk, conf_blk, pconf[:, s:e])
            block_steps.append(_snap(si, nb, s, e, x, x0_snap, pconf))
            nfe += 1

        history.append({'block_id': nb, 'steps': block_steps})
        del context_kv

    return x, nfe, history


print('generate_with_collection_prune() defined — 1-forward per block')
print(f'  suffix_mode: full / tail / window')

## 6. 跑实验 + 生成 HTML

每种模式跑完 50 个 sample 后立即生成 HTML 并释放内存。

In [ ]:
from viz_static import process_sample, generate_multi_html
import time

OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, 'viz_suffix_prune')
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary = []   # (mode, acc, total_nfe, time, html_path, html_size_mb)
all_gen_texts = {}  # mode → list of generated texts

for mode in MODES:
    print(f'\n{"="*60}')
    print(f'  suffix_mode = {mode}')
    if mode == 'tail':
        print(f'  warm-up 输入: prefix + block + 1 tail token')
    elif mode == 'window':
        print(f'  warm-up 输入: prefix + block + {WINDOW_BLOCKS}×{BLOCK_LENGTH} window + tail')
    else:
        print(f'  warm-up 输入: 完整序列')
    print(f'{"="*60}')

    all_samples = []
    total_nfe = 0
    correct = 0
    gen_texts = []
    t0 = time.time()

    for i in range(LIMIT):
        prompt = prompts[i]

        x, nfe, history = generate_with_collection_prune(
            model, prompt,
            steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
            temperature=0.0, threshold=THRESHOLD, mask_id=MASK_ID,
            suffix_mode=mode,
            window_blocks=WINDOW_BLOCKS,
        )
        total_nfe += nfe

        gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
        for stop in ['Question:', '\n\nQuestion']:
            if stop in gen_text:
                gen_text = gen_text.split(stop)[0]
        gen_texts.append(gen_text)

        gen_ans = extract_answer(gen_text)
        ref_ans = extract_answer(ref_answers[i])
        is_correct = gen_ans is not None and ref_ans is not None and gen_ans == ref_ans
        if is_correct:
            correct += 1

        sample_json = process_sample(
            history, tokenizer,
            question=questions[i],
            gen_answer='{} {}'.format(gen_ans, '\u2713' if is_correct else '\u2717 (ref: {})'.format(ref_ans)),
            ref_answer=ref_ans,
        )
        all_samples.append(sample_json)

        if (i + 1) % 10 == 0 or i == LIMIT - 1:
            elapsed = time.time() - t0
            print(f'  [{i+1}/{LIMIT}] NFE={nfe:3d}  acc={correct}/{i+1}  ({elapsed:.0f}s)')

    elapsed = time.time() - t0
    acc = correct / LIMIT
    all_gen_texts[mode] = gen_texts

    # Generate HTML
    html = generate_multi_html(
        all_samples,
        title=f'GSM8K Suffix Pruning — {mode} (acc={acc:.2%}, NFE={total_nfe})',
    )
    html_path = os.path.join(OUTPUT_DIR, f'viz_suffix_{mode}.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html)
    size_mb = os.path.getsize(html_path) / 1024 / 1024

    summary.append((mode, acc, total_nfe, elapsed, html_path, size_mb))
    print(f'  \u2192 acc={acc:.2%}  NFE={total_nfe}  time={elapsed:.0f}s  HTML={size_mb:.1f}MB')

    del all_samples, html, history
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('All done!')

## 7. 汇总对比

In [ ]:
import pandas as pd

df = pd.DataFrame(summary, columns=['mode', 'accuracy', 'total_nfe', 'time_sec', 'html_path', 'html_size_mb'])
display(df[['mode', 'accuracy', 'total_nfe', 'time_sec', 'html_size_mb']])

# Text identity check
if 'full' in all_gen_texts:
    bl = all_gen_texts['full']
    for mode in ['tail', 'window']:
        if mode in all_gen_texts:
            tl = all_gen_texts[mode]
            match_count = sum(1 for a, b in zip(bl, tl) if a == b)
            print(f'\n=== full vs {mode} ===')
            print(f'  文本完全一致: {match_count}/{LIMIT}')

print(f'\nHTML files saved to: {OUTPUT_DIR}/')
for _, row in df.iterrows():
    print(f'  {row["mode"]:8s}  acc={row["accuracy"]:.2%}  \u2192 {os.path.basename(row["html_path"])} ({row["html_size_mb"]:.1f} MB)')
print(f'\n下载 viz_suffix_prune/ 目录到本地，用浏览器打开 HTML 即可浏览 denoising 过程')

## 8. Accuracy & NFE 对比图

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'full': '#2196F3', 'tail': '#FF5722', 'window': '#4CAF50'}

# Accuracy
ax = axes[0]
bars = ax.bar(df['mode'], df['accuracy'], color=[colors[m] for m in df['mode']], width=0.5)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Mode', fontweight='bold')
ax.set_ylim(0, max(df['accuracy']) * 1.2)
for bar, acc in zip(bars, df['accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.2%}', ha='center', fontsize=11, fontweight='bold')

# NFE
ax = axes[1]
bars = ax.bar(df['mode'], df['total_nfe'], color=[colors[m] for m in df['mode']], width=0.5)
ax.set_ylabel('Total NFE (50 samples)')
ax.set_title('NFE by Mode', fontweight='bold')
for bar, nfe in zip(bars, df['total_nfe']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(nfe)), ha='center', fontsize=11, fontweight='bold')

# Time
ax = axes[2]
bars = ax.bar(df['mode'], df['time_sec'], color=[colors[m] for m in df['mode']], width=0.5)
ax.set_ylabel('Time (seconds)')
ax.set_title('Time by Mode', fontweight='bold')
for bar, t in zip(bars, df['time_sec']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{t:.0f}s', ha='center', fontsize=11, fontweight='bold')

fig.suptitle(f'Suffix Pruning Comparison (GSM8K, {LIMIT} samples)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'suffix_prune_summary.png'), dpi=150, bbox_inches='tight')
plt.show()